# eda

_Adam_

In [ ]:

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))



In [ ]:

cols = ['_AGE_G', '_AGEG5YR',
'_AGE80', '_SEX',
'_IMPRACE', '_EDUCAG',
'EMPLOY1', 'MARITAL',
'CHILDREN', 'VETERAN3',
'RECENT_CHECKUP',
'GOT_FLUSHOT',
'HAS_EXERCISE','_LLCPWT_POOLED']


df = pd.read_parquet('/kaggle/input/datasets/ajenks/brfss-2020-2024-cleaned-and-weighted/brfss_2020_2024_pooled_eda.parquet', columns = cols)

In [ ]:
df.head()

Missing data values for columns

EDUCA - 9

EMPLOY1 - 9

MARITAL - 9

_AGEG5YR - 14

_SEX - 1 is male, 2 is female

VETERAN - 1 is yes, 2 is no



In [ ]:
import numpy as np

df['EMPLOY1'] = df['EMPLOY1'].replace(9, np.nan)
# df['_EDUCAG'] = df['_EDUCAG'].replace(9, np.nan)
df['MARITAL'] = df['MARITAL'].replace(9, np.nan)
df['_AGEG5YR'] = df['_AGEG5YR'].replace(14, np.nan)

In [ ]:
for col in ['EMPLOY1', '_EDUCAG', 'MARITAL', '_AGEG5YR']:
    print(col)
    print(df[col].value_counts(dropna=False))
    print()

In [ ]:
df.head(), df.shape

In [ ]:
df = df.dropna(subset=['_EDUCAG','EMPLOY1','MARITAL','_AGEG5YR'])

In [ ]:
for col in ['EMPLOY1', '_EDUCAG', 'MARITAL', '_AGEG5YR','_LLCPWT_POOLED']:
    print(col)
    print(df[col].value_counts(dropna=False))
    print()

In [ ]:
df.shape

Need to do some mapping to make sense of these categorical variables

In [ ]:
employment_mapping = {
    1: "Employed",
    2: "Employed",
    3: "Unemployed",
    4: "Unemployed",
    5: "Homemaker",
    6: "Student",
    7: "Retired",
    8: "Unable to work",
}

age_mapping = {
    1: '18 to 24',
    2: '25 to 34',
    3: '35 to 44',
    4: '45 to 54',
    5: '55 to 64',
    6: '65 and older'
    
}

education_grouped = {
    1: "Less than High School",
    2: "Graduated High School",
    3: "Attended College",
    4: "Graduated College",
}

race_mapping = {
    1: 'White',
    2: 'Black',
    3: 'Asian',
    4: 'American Indian',
    5: 'Hispanic',
    6: 'Other'
}
sex_mapping = {
    1:'Male',
    2: 'Female'
}

df['_EDUCAG'] = df['_EDUCAG'].map(education_grouped)

df['_SEX'] = df['_SEX'].map(sex_mapping)

df['_AGE_G'] = df['_AGE_G'].map(age_mapping)

df['EMPLOY1'] = df['EMPLOY1'].map(employment_mapping)

df['_IMPRACE'] = df['_IMPRACE'].map(race_mapping)

df['EMPLOY1'] = df['EMPLOY1'].fillna('Missing')

In [ ]:


df['_EDUCAG'] = pd.Categorical(
    df['_EDUCAG'],
    categories=[
        "Graduated College",
        "Attended College",
        "Graduated High School",
        "Less than High School"
    ],
    ordered=True
)

In [ ]:
df['_EDUCAG'].value_counts()

In [ ]:
df['VETERAN3']= df['VETERAN3'].map({1:1,2:0})

In [ ]:
df['VETERAN3'].value_counts()

Start by making categories out of our variables

In [ ]:
df.head()

**The first relationship I'd like to look at is how employment and education status affects exercise activity**

In [ ]:
grouped = (
    df.groupby(['_EDUCAG', 'EMPLOY1'])['HAS_EXERCISE']
    .mean()
    .reset_index()
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
weight_col ='_LLCPWT_POOLED'

# Physical Activity plot df
exer_plot_df = df[['HAS_EXERCISE', '_IMPRACE', '_EDUCAG', weight_col]].copy()

exer_plot_df['HAS_EXERCISE'] = pd.to_numeric(exer_plot_df['HAS_EXERCISE'], errors='coerce')
exer_plot_df[weight_col] = pd.to_numeric(exer_plot_df[weight_col], errors='coerce')

exer_plot_df = exer_plot_df.dropna(subset=['HAS_EXERCISE', '_IMPRACE', '_EDUCAG', weight_col])
exer_plot_df = exer_plot_df[exer_plot_df[weight_col] > 0]

# Checkup plot df
checkup_plot_df = df[['RECENT_CHECKUP', '_IMPRACE', '_EDUCAG', weight_col]].copy()

checkup_plot_df['RECENT_CHECKUP'] = pd.to_numeric(checkup_plot_df['RECENT_CHECKUP'], errors='coerce')
checkup_plot_df[weight_col] = pd.to_numeric(checkup_plot_df[weight_col], errors='coerce')

checkup_plot_df = checkup_plot_df.dropna(subset=['RECENT_CHECKUP', '_IMPRACE', '_EDUCAG', weight_col])
checkup_plot_df = checkup_plot_df[checkup_plot_df[weight_col] > 0]

# Flu shot plot df

flu_shot_df = df[['GOT_FLUSHOT', '_IMPRACE', '_EDUCAG', weight_col]].copy()

flu_shot_df['GOT_FLUSHOT'] = pd.to_numeric(flu_shot_df['GOT_FLUSHOT'], errors='coerce')
flu_shot_df[weight_col] = pd.to_numeric(flu_shot_df[weight_col], errors='coerce')

flu_shot_df = flu_shot_df.dropna()
flu_shot_df = flu_shot_df[flu_shot_df[weight_col] > 0]




In [ ]:
exercise_pivot = exer_plot_df.pivot_table(
    values='HAS_EXERCISE',
    index='_IMPRACE',
    columns='_EDUCAG',
    aggfunc=lambda x: np.average(x, weights=exer_plot_df.loc[x.index, weight_col]),
    observed=False
)

checkup_pivot = checkup_plot_df.pivot_table(
    values='RECENT_CHECKUP',
    index='_IMPRACE',
    columns='_EDUCAG',
    aggfunc=lambda x: np.average(x, weights=checkup_plot_df.loc[x.index, weight_col]),
    observed=False
)

flu_shot_pivot = flu_shot_df.pivot_table(
    values='GOT_FLUSHOT',
    index='_IMPRACE',
    columns='_EDUCAG',
    aggfunc=lambda x: np.average(x, weights=flu_shot_df.loc[x.index, weight_col]),
    observed=False
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 10), sharey=True, dpi = 300)

sns.heatmap(exercise_pivot, annot=True, cmap='coolwarm',ax = axes[0])

axes[0].set_xlabel("Education Level")
axes[0].set_ylabel("Race")
axes[0].set_title("Physically Active Individuals by Education and Race")
axes[0].tick_params(axis='x', rotation=30)
axes[0].tick_params(axis='y', rotation=0)

sns.heatmap(checkup_pivot,annot=True,cmap="coolwarm",ax=axes[1])

axes[1].set_title("Recent Checkup by Education and Race")
axes[1].set_xlabel("Education Level")
axes[1].set_ylabel("")
axes[1].tick_params(axis='x', rotation=30)
axes[1].tick_params(axis='y', rotation=0)

sns.heatmap(flu_shot_pivot,annot=True,cmap="coolwarm",ax=axes[2])
axes[2].set_title('Flu Shot Status by Education and Race')
axes[2].set_xlabel('Education Level')
axes[2].set_ylabel('')
axes[2].tick_params(axis = 'x', rotation = 30)
axes[2].tick_params(axis = 'y', rotation = 0)



plt.tight_layout()
plt.show()

In [ ]:
age_race_pivot = df.pivot_table(
    values = 'RECENT_CHECKUP',
    index = '_IMPRACE',
    columns = '_EDUCAG',
    aggfunc = 'mean'


)

In [ ]:
grouped = (
    df.groupby(['_AGE_G', 'RECENT_CHECKUP'])
      .size()
      .reset_index(name='count')
)

grouped['proportion'] = (
    grouped.groupby('_AGE_G')['count']
           .transform(lambda x: x / x.sum())
)

pivot = grouped.pivot(
    index='_AGE_G',
    columns='RECENT_CHECKUP',
    values='proportion'
)

In [ ]:
pivot.plot(kind='bar')

plt.title('Recent Checkup by Age Group')
plt.xlabel('Age Group')
plt.ylabel('Proportion')
plt.legend(['No Checkup', 'Recent Checkup'])
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
sex_grouped = (
    df.groupby(['_SEX','GOT_FLUSHOT'])
    .size()
    .reset_index(name = 'count')
)

sex_grouped['proportion'] = (
    sex_grouped.groupby('_SEX')['count']
           .transform(lambda x: x / x.sum())
)

pivot = sex_grouped.pivot(
    index='_SEX',
    columns='GOT_FLUSHOT',
    values='proportion'
)

In [ ]:
pivot.plot(kind='bar')

plt.title('Flu Shot Status ')
plt.xlabel('Age Group')
plt.ylabel('Proportion')
plt.legend(['No Checkup', 'Recent Checkup'])
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
pivot = df.groupby(['_AGE_G', '_SEX'])['RECENT_CHECKUP'].mean().unstack()

pivot.plot(kind='bar', stacked=False, figsize=(10,6))

plt.title('Stacked Recent Checkup Rates by Age and Sex')
plt.xlabel('Age Group')
plt.ylabel('Proportion')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
def weighted_mean(x, weights):
    return (x * weights).sum() / weights.sum()

In [ ]:
grouped = (
    df.groupby(['_AGE_G', '_SEX'])
    .apply(lambda g: weighted_mean(g['RECENT_CHECKUP'], g[weight_col]))
    .reset_index(name='weighted_prop')
)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))
sns.barplot(data=grouped, x='_AGE_G', y='weighted_prop', hue='_SEX')

plt.title('Weighted Recent Checkup Rates by Age and Sex')
plt.xlabel('Age Group')
plt.ylabel('Proportion with Recent Checkup')
plt.legend(title='Sex')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
activity_emp = (
    df.groupby('EMPLOY1')
    .apply(lambda g: weighted_mean(g['HAS_EXERCISE'], g[weight_col]))
    .reset_index(name='weighted_prop')
)

activity_emp = activity_emp.sort_values('weighted_prop')

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,6))

plt.scatter(activity_emp['weighted_prop'], activity_emp['EMPLOY1'])

plt.xlabel('Proportion Physically Active (Weighted)')
plt.ylabel('Employment Status')
plt.title('Physical Activity by Employment Status (Weighted)')

plt.grid(axis='x', linestyle='--', alpha=0.7)

for i, v in enumerate(activity_emp['weighted_prop']):
    plt.text(v + 0.005, i, f"{v:.2f}", va='center')


plt.tight_layout()
plt.show()